# Benchmarking with `alf-benchmark`

This tutorial shows how to (1) run a versioned benchmark suite, (2) inspect the
leaderboard and active-learning curves, and (3) register your own model and try
to beat the baseline.

It uses the CPU-friendly `alf-protein-v1` suite (bundled GFP dataset, no GPU or
download required).

## 1. Run a suite

From the command line:

```bash
alf-bench run benchmark/alf_benchmark/suites/alf_protein_v1.yaml
alf-bench aggregate runs/alf_protein_v1 --markdown LEADERBOARD.md
```

Or from Python, which is what the rest of this notebook uses:

In [ ]:
from alf_benchmark import BenchmarkResults, load_run_config, run_benchmark

run_config = load_run_config("../../benchmark/alf_benchmark/suites/alf_protein_v1.yaml")
manifests = run_benchmark(run_config)
print(f"{sum(m.status == 'completed' for m in manifests)} / {len(manifests)} completed")

## 2. Inspect the leaderboard and curves

Results load into a tidy long-form table; `leaderboard()` ranks methods on each
problem's primary metric (mean ± bootstrap CI).

In [ ]:
results = BenchmarkResults.from_dir(run_config.output_dir)
results.leaderboard()

In [ ]:
from alf_benchmark.plotting import plot_curves

# Simple regret is logged directly; best-found-so-far is derived.
plot_curves(results, metric="optimizer/regret")
plot_curves(results, metric="derived/best_found");

## 3. Register your own model and beat the baseline

Subclass `BaseModel`, register it under the `alf.models` group, and reference it
by name in a method. (In a real package you would expose it as a `pyproject.toml`
entry point instead of registering in-process — then no import of `alf-benchmark`
is needed.)

In [ ]:
import numpy as np
from alf_core.dataclasses import Predictions
from alf_core.model.base_model import BaseModel
from alf_benchmark import register


@register("alf.models", "mean_predictor")
class MeanPredictor(BaseModel):
    """Trivial baseline: predicts the training-set mean for every candidate."""

    def __init__(self, name: str = "mean_predictor") -> None:
        self.name = name
        self._mean = 0.0

    def featurise(self, inputs):
        return None

    def train(self, train_data, val_data=None):
        self._mean = float(np.mean(train_data.labels))

    def predict(self, candidate_points):
        return Predictions(means=np.full(len(candidate_points), self._mean))

    def sample(self, condition=None):
        raise NotImplementedError

In [ ]:
from alf_benchmark import (
    BenchmarkMethod,
    BenchmarkRunner,
    BenchmarkSuite,
    ComponentSpec,
    MethodConfig,
    TaskConfig,
)

suite = BenchmarkSuite.from_configs(
    run_config.suite_name, run_config.suite_version, run_config.problems
)
my_method = BenchmarkMethod(
    MethodConfig(
        name="mean_predictor_greedy",
        family="design",
        surrogate=ComponentSpec(name="mean_predictor"),
        acquisition=ComponentSpec(name="greedy"),
        search=ComponentSpec(name="dataset_search"),
        task=TaskConfig(num_acq_rounds=3, acq_batch_size=50),
    )
)
BenchmarkRunner().run(suite, [my_method], run_config.output_dir)

# Reload and compare against the committed baselines.
BenchmarkResults.from_dir(run_config.output_dir).leaderboard()

The `mean_predictor` ignores sequence information, so a real surrogate (CNN, GP)
should rank above it on `optimizer/regret`. Swap in your own model and see where
it lands.